In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Notebook lives in credit-default/notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd().parent
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
FIG_DIR      = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR    = PROJECT_ROOT / "outputs" / "tables"

RAW_LOANS = RAW_DIR / "loan.csv"
assert RAW_LOANS.exists(), f"Not found: {RAW_LOANS}"

print(f"Project root: {PROJECT_ROOT}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pa.__version__}")
print(f"loan.csv: {RAW_LOANS.stat().st_size / 1e9:.2f} GB")

Project root: /Users/sourodeeproy/Desktop/credit-default
pandas 3.1.0 | numpy 2.5.4 | pyarrow 26.0.0
loan.csv: 1.68 GB


In [2]:
# 1. Header only: how many columns, and what are they?
header = pd.read_csv(RAW_LOANS, nrows=0)
print(f"Columns: {len(header.columns)}")

# 2. Read a 100k-row sample and measure its real in-memory size
t0 = time.time()
sample = pd.read_csv(RAW_LOANS, nrows=100_000, low_memory=False)
print(f"Read 100k rows in {time.time() - t0:.1f}s")

mem_sample = sample.memory_usage(deep=True).sum()
TOTAL_ROWS_APPROX = 2_260_701          # from v1; confirmed exactly during conversion
est_full_gb = mem_sample * TOTAL_ROWS_APPROX / len(sample) / 1e9

print(f"Sample memory:            {mem_sample / 1e6:,.0f} MB")
print(f"Projected full-file load: {est_full_gb:,.1f} GB (excluding parsing overhead)")
print("\nColumn dtypes in sample:")
print(sample.dtypes.astype(str).value_counts())

Columns: 151
Read 100k rows in 0.3s
Sample memory:            143 MB
Projected full-file load: 3.2 GB (excluding parsing overhead)

Column dtypes in sample:
float64    114
str         36
int64        1
Name: count, dtype: int64


In [3]:
CHUNK_ROWS = 250_000
cols = header.columns.tolist()

n_rows       = 0
non_null     = pd.Series(0, index=cols, dtype="int64")
non_numeric  = pd.Series(0, index=cols, dtype="int64")   # non-null values that fail numeric parsing

t0 = time.time()
reader = pd.read_csv(RAW_LOANS, dtype=str, chunksize=CHUNK_ROWS)
for i, chunk in enumerate(reader, start=1):
    n_rows   += len(chunk)
    non_null += chunk.notna().sum()
    for c in cols:
        s = chunk[c].dropna()
        if len(s):
            non_numeric[c] += pd.to_numeric(s, errors="coerce").isna().sum()
    print(f"  chunk {i:>2}: {n_rows:>10,} rows scanned  ({time.time() - t0:5.1f}s)")

# id is an identifier, never a number to compute with, so keep it as text regardless
numeric_cols = [c for c in cols if non_numeric[c] == 0 and c != "id"]
text_cols    = [c for c in cols if c not in numeric_cols]

print(f"\nTotal rows in file: {n_rows:,}")
print(f"Numeric columns: {len(numeric_cols)} | Text columns: {len(text_cols)}")

# Where did the 100k-row sample guess wrong?
sample_numeric = set(sample.select_dtypes(include="number").columns)
disagree = sorted((sample_numeric - set(numeric_cols)) | (set(numeric_cols) - sample_numeric))
print(f"\nColumns where the sample's guess differs from the full-file schema: {disagree}")

# Mostly-numeric columns that contain a few text values: worth a look
suspicious = non_numeric[(non_numeric > 0) & (non_numeric < 0.01 * non_null)]
print("\nMostly numeric but with some non-numeric values:")
print(suspicious.to_string() if len(suspicious) else "  none")

  chunk  1:    250,000 rows scanned  (  8.1s)
  chunk  2:    500,000 rows scanned  ( 16.9s)
  chunk  3:    750,000 rows scanned  ( 26.0s)
  chunk  4:  1,000,000 rows scanned  ( 34.8s)
  chunk  5:  1,250,000 rows scanned  ( 43.2s)
  chunk  6:  1,500,000 rows scanned  ( 51.6s)
  chunk  7:  1,750,000 rows scanned  ( 60.0s)
  chunk  8:  2,000,000 rows scanned  ( 68.1s)
  chunk  9:  2,250,000 rows scanned  ( 77.1s)
  chunk 10:  2,260,701 rows scanned  ( 77.6s)

Total rows in file: 2,260,701
Numeric columns: 113 | Text columns: 38

Columns where the sample's guess differs from the full-file schema: ['id', 'sec_app_earliest_cr_line']

Mostly numeric but with some non-numeric values:
id    33


In [4]:
RAW_PARQUET = INTERIM_DIR / "loans_raw.parquet"

# Fixed Arrow schema decided by Pass 1: numbers as float64, everything else as string
schema = pa.schema(
    [(c, pa.float64()) if c in numeric_cols else (c, pa.string()) for c in cols]
)

t0 = time.time()
rows_written = 0
writer = pq.ParquetWriter(RAW_PARQUET, schema, compression="zstd")
try:
    for i, chunk in enumerate(pd.read_csv(RAW_LOANS, dtype=str, chunksize=CHUNK_ROWS), start=1):
        # Apply the fixed schema: parse numerics, force float64 so every chunk matches exactly
        for c in numeric_cols:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce").astype("float64")
        table = pa.Table.from_pandas(chunk, schema=schema, preserve_index=False)
        writer.write_table(table)
        rows_written += len(chunk)
        print(f"  chunk {i:>2}: {rows_written:>10,} rows written  ({time.time() - t0:5.1f}s)")
finally:
    writer.close()          # always close, or the file footer is never written and the file is unreadable

# Save the schema decision alongside the data, for reproducibility
with open(INTERIM_DIR / "loans_raw_schema.json", "w") as f:
    json.dump({"numeric_cols": numeric_cols, "text_cols": text_cols}, f, indent=2)

# Verify: row count from file metadata, size, and how fast a column subset now loads
meta = pq.read_metadata(RAW_PARQUET)
print(f"\nRows in Parquet:  {meta.num_rows:,}  (CSV had {n_rows:,})")
print(f"Columns:          {meta.num_columns}")
print(f"File size:        {RAW_PARQUET.stat().st_size / 1e9:.2f} GB  (CSV: {RAW_LOANS.stat().st_size / 1e9:.2f} GB)")

t0 = time.time()
test = pd.read_parquet(RAW_PARQUET, columns=["id", "issue_d", "loan_status", "loan_amnt"])
print(f"Loaded 4 columns x {len(test):,} rows in {time.time() - t0:.1f}s")
assert meta.num_rows == n_rows, "Row count mismatch between CSV and Parquet"

  chunk  1:    250,000 rows written  (  7.6s)
  chunk  2:    500,000 rows written  ( 14.9s)
  chunk  3:    750,000 rows written  ( 23.2s)
  chunk  4:  1,000,000 rows written  ( 31.3s)
  chunk  5:  1,250,000 rows written  ( 38.4s)
  chunk  6:  1,500,000 rows written  ( 46.0s)
  chunk  7:  1,750,000 rows written  ( 53.3s)
  chunk  8:  2,000,000 rows written  ( 60.4s)
  chunk  9:  2,250,000 rows written  ( 67.7s)
  chunk 10:  2,260,701 rows written  ( 68.1s)

Rows in Parquet:  2,260,701  (CSV had 2,260,701)
Columns:          151
File size:        0.29 GB  (CSV: 1.68 GB)
Loaded 4 columns x 2,260,701 rows in 3.7s
